
## 03 — Feature Engineering & Derived Business Columns
Purpose: Derive retention-relevant business columns and NumPy-vectorised sequential/ rolling features on top of the merged analytical table, per PRD Section 9 (PR7). This notebook covers PR7 only (PR8–PR10 continue in later cells / later PRs).

**Inputs**:

- data/processed/streakforge_merged.csv
- data/interim/subscription_renewal_records_clean.csv (member-level truth for churn/spend)
- data/interim/streak_history_episodes_clean.csv (member-level truth for streak recovery)
- 
**Outputs**:
- data/processed/streakforge_features.csv — merged table + 20 derived feature columns

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

PROCESSED_DIR = Path("../data/processed")
INTERIM_DIR = Path("../data/interim")

AS_OF_DATE = pd.Timestamp("2026-08-14")     # same observation cutoff as notebooks 01 & 02
GRACE_PERIOD_DAYS = 30                      # matches the spec's lapse/grace window

DATE_COLS = ["event_datetime", "date_of_birth", "join_date", "billing_cycle_start",
             "billing_cycle_end", "streak_start_date", "streak_end_date"]

merged = pd.read_csv(PROCESSED_DIR / "streakforge_merged.csv", parse_dates=DATE_COLS, low_memory=False)
subs = pd.read_csv(INTERIM_DIR / "subscription_renewal_records_clean.csv",
                    parse_dates=["billing_cycle_start", "billing_cycle_end"])
streak = pd.read_csv(INTERIM_DIR / "streak_history_episodes_clean.csv",
                      parse_dates=["streak_start_date", "streak_end_date"])

print("merged:", merged.shape)

merged: (206732, 60)



## 7.1 Derived business columns
Straightforward vectorised pandas/NumPy arithmetic and boolean masks — no per-row .apply(). These are the "cheap" derived columns: date-arithmetic, ratios, and flags that only need the event's own row (plus what the temporal joins already attached to it in notebook 02).

In [2]:
merged["is_gym_session"] = merged["event_source"].eq("Gym Checkin")
merged["is_app_event"]   = merged["event_source"].eq("App Event")

# --- membership tenure / plan timing ---
merged["tenure_days_at_event"]  = (merged["event_datetime"] - merged["join_date"]).dt.days
merged["is_new_member_event"]   = merged["tenure_days_at_event"] <= 30

merged["days_since_plan_start"] = (merged["event_datetime"] - merged["billing_cycle_start"]).dt.days
merged["plan_duration_days"]    = (merged["billing_cycle_end"] - merged["billing_cycle_start"]).dt.days
merged["discount_amount_inr"]   = merged["amount_paid_inr"] * (merged["discount_pct"] / 100)
merged["price_per_day_inr"] = np.where(
    merged["plan_duration_days"] > 0,
    merged["amount_paid_inr"] / merged["plan_duration_days"],
    np.nan,
)

# --- time-of-day / day-of-week business flags ---
merged["is_weekend_event"]     = merged["event_day_of_week"].isin(["Saturday", "Sunday"])
merged["is_early_morning_slot"] = merged["event_hour"].between(5, 7)

# --- streak position: how many days into the matched streak is this event ---
merged["current_streak_day_number"] = np.where(
    merged["is_within_active_streak"],
    (merged["event_datetime"] - merged["streak_start_date"]).dt.days + 1,
    np.nan,
)

print(merged[["event_source", "is_gym_session", "tenure_days_at_event",
              "days_since_plan_start", "discount_amount_inr", "price_per_day_inr",
              "is_weekend_event", "current_streak_day_number"]].head())

  event_source  is_gym_session  tenure_days_at_event  days_since_plan_start  discount_amount_inr  price_per_day_inr  is_weekend_event  \
0    App Event           False                     0                      0                  0.0         204.725275              True   
1    App Event           False                     1                      1               1244.0         273.406593              True   
2  Gym Checkin            True                     1                      1                417.5          55.666667              True   
3  Gym Checkin            True                     1                      1                  0.0         126.098901              True   
4    App Event           False                     1                      1                176.0         117.333333              True   

   current_streak_day_number  
0                        NaN  
1                        NaN  
2                        NaN  
3                        NaN  
4                     

## 7.2 NumPy vectorised computation workflow
Per-member sequential features (gaps between sessions, trailing-window counts) can't be derived from a single row — they need that member's full event history. Rather than a naive nested loop, these use np.searchsorted on a sorted int64-ns timestamp array: an O(n log n) binary-search rolling-window count, computed once per member via groupby(...).apply(...), with all the actual arithmetic done in NumPy inside each call — not row-by-row Python.

In [3]:
def rolling_trailing_count(sorted_ns: np.ndarray, window_days: int) -> np.ndarray:
    """
    For each timestamp in a *sorted* int64-ns array, count how many timestamps
    (including itself) fall within `window_days` days before it.
    Pure NumPy binary search — works on irregular, non-fixed-frequency event
    timestamps where pandas' `.rolling('Nd')` can't be used directly on an
    unindexed array.
    """
    if sorted_ns.size == 0:
        return np.array([], dtype=np.int64)
    window_ns = np.int64(window_days) * 24 * 3600 * 1_000_000_000
    lower_bound = sorted_ns - window_ns
    left_idx = np.searchsorted(sorted_ns, lower_bound, side="left")
    right_idx = np.searchsorted(sorted_ns, sorted_ns, side="right")
    return right_idx - left_idx


def compute_member_sequence_features(group: pd.DataFrame) -> pd.DataFrame:
    """Runs once per member; all math below is vectorised NumPy on that member's
    event array. `group_keys=False` + grouping on the index (not the member_id
    *column*) keeps member_id available inside `group`, since pandas 3.x now
    drops the literal grouping column from the frame passed into .apply()."""
    g = group.sort_values("event_datetime")
    ns = g["event_datetime"].values.astype("datetime64[ns]").astype(np.int64)

    # --- days_since_last_session (gym check-ins only) ---
    is_gym = g["is_gym_session"].values
    gym_ns = ns[is_gym]
    gap_days_gym = np.full(is_gym.sum(), np.nan)
    if gym_ns.size > 1:
        gap_days_gym[1:] = np.diff(gym_ns) / (24 * 3600 * 1_000_000_000)
    days_since_last_session = np.full(len(g), np.nan)
    days_since_last_session[is_gym] = gap_days_gym

    # --- session_consistency_score: coefficient of variation of gym-session gaps ---
    valid_gaps = gap_days_gym[~np.isnan(gap_days_gym)]
    consistency_score = (
        valid_gaps.std() / valid_gaps.mean()
        if valid_gaps.size >= 2 and valid_gaps.mean() > 0 else np.nan
    )

    # --- sessions_per_week_rolling_4wk: trailing 28-day gym-session count, as-of each event ---
    sessions_28d = np.zeros(len(g), dtype=np.int64)
    if gym_ns.size > 0:
        sessions_28d[is_gym] = rolling_trailing_count(gym_ns, window_days=28)
    sessions_per_week_rolling_4wk = sessions_28d / 4.0

    # --- app_opens_last_7d / notifications_clicked_last_7d: trailing 7-day counts ---
    is_app = g["is_app_event"].values
    opens_mask = (g["event_type"].values == "App Opened–No Action") & is_app
    notif_mask = (g["event_type"].values == "Push Notification Opened") & is_app

    opens_7d = np.zeros(len(g), dtype=np.int64)
    notif_7d = np.zeros(len(g), dtype=np.int64)
    if is_app.any():
        app_ns = ns[is_app]
        opens_7d[is_app] = rolling_trailing_count(app_ns, 7)
        notif_7d[is_app] = rolling_trailing_count(app_ns, 7)
    opens_7d = np.where(opens_mask | ~is_app, opens_7d, 0)
    notif_7d = np.where(notif_mask | ~is_app, notif_7d, 0)

    return g.assign(
        days_since_last_session=days_since_last_session,
        session_consistency_score=consistency_score,
        sessions_per_week_rolling_4wk=sessions_per_week_rolling_4wk,
        app_opens_last_7d=opens_7d,
        notifications_clicked_last_7d=notif_7d,
    )


print("Computing per-member sequential NumPy features (206,732 rows)...")
merged = (
    merged.set_index("member_id", drop=False)
    .groupby(level=0, group_keys=False)
    .apply(compute_member_sequence_features)
    .reset_index(drop=True)
)

# sanity check: every "App Opened" event should show a 7d count >= 1 (it counts itself)
opens = merged.loc[merged["event_type"].eq("App Opened–No Action")]
print("\napp_opens_last_7d on actual app-open rows (should be >=1 everywhere):")
print(opens["app_opens_last_7d"].value_counts().sort_index())

Computing per-member sequential NumPy features (206,732 rows)...

app_opens_last_7d on actual app-open rows (should be >=1 everywhere):
app_opens_last_7d
1    13290
2      800
3       45
4        3
5        1
Name: count, dtype: int64


## 7.3 Member-level derived features
Some retention-relevant columns are true member-level facts (churn status, ₹-per-session spend, engagement trend) rather than per-event facts. These are computed once per member — using the raw subscription/streak tables for the "ground truth" versions, since the as-of joined event view is deliberately skewed toward Lapsed/Grace states (per PR6's finding) — then broadcast back onto every event row for that member.

In [4]:
# --- is_churned: from Fact_Subscriptions directly, not the as-of joined event view ---
last_cycle_end = subs.groupby("member_id")["billing_cycle_end"].max()
churn_cutoff = AS_OF_DATE - pd.Timedelta(days=GRACE_PERIOD_DAYS)
is_churned_map = (last_cycle_end < churn_cutoff).reindex(merged["member_id"].unique()).fillna(True)
merged["is_churned"] = merged["member_id"].map(is_churned_map)

# --- plan_to_engagement_ratio: ₹ paid per logged gym session (surfaces "paying, not showing up") ---
total_paid = subs.groupby("member_id")["amount_paid_inr"].sum()
total_sessions = merged.loc[merged["is_gym_session"]].groupby("member_id").size()
plan_to_engagement_ratio = (total_paid / total_sessions.replace(0, np.nan)).rename("plan_to_engagement_ratio")
merged = merged.merge(plan_to_engagement_ratio, on="member_id", how="left")

# --- engagement_trend_slope: NumPy polyfit slope of weekly gym-session counts, trailing 8 weeks ---
def member_trend_slope(member_events: pd.DataFrame) -> float:
    gym_dates = member_events.loc[member_events["is_gym_session"], "event_datetime"]
    if len(gym_dates) < 3:
        return np.nan
    last_date = member_events["event_datetime"].max()
    windowed = gym_dates[gym_dates >= last_date - pd.Timedelta(weeks=8)]
    if windowed.empty:
        return np.nan
    week_bucket = (last_date - windowed).dt.days // 7
    weekly_counts = week_bucket.value_counts().reindex(range(8), fill_value=0).sort_index()
    x = np.arange(len(weekly_counts))[::-1]          # oldest week -> most recent week
    y = weekly_counts.values.astype(float)
    if np.all(y == y[0]):                              # flat line -> polyfit is degenerate
        return 0.0
    slope, _ = np.polyfit(x, y, deg=1)
    return slope

trend_slopes = (
    merged.groupby("member_id")[["event_datetime", "is_gym_session"]]
    .apply(member_trend_slope, include_groups=False)
    .rename("engagement_trend_slope")
)
merged = merged.merge(trend_slopes, on="member_id", how="left")

print(merged[["member_id", "is_churned", "plan_to_engagement_ratio", "engagement_trend_slope"]]
      .drop_duplicates("member_id").head(8))

      member_id  is_churned  plan_to_engagement_ratio  engagement_trend_slope
0    MBR-005512        True                931.500000                0.059524
33   MBR-000580        True                956.923077                     NaN
97   MBR-011948        True                417.500000                     NaN
120  MBR-000791        True               5695.714286               -0.035714
136  MBR-009955        True                160.000000                     NaN
186  MBR-009562        True                234.090909                     NaN
227  MBR-000933        True                500.526316                     NaN
259  MBR-014767        True                308.000000                     NaN


## 7.4 streak_break_recovery_flag
Computed at Fact_StreakHistory grain: did the member log a fresh gym check-in within 14 days of a finished streak ending? Ongoing streaks (no end date yet) have nothing to recover from, so they're left null rather than defaulted to False. Same NumPy searchsorted pattern as 7.2, applied per member.

In [5]:
gym_checkins_sorted = (
    merged.loc[merged["is_gym_session"], ["member_id", "event_datetime"]]
    .sort_values(["member_id", "event_datetime"])
)
checkin_ns_by_member = {
    mid: grp["event_datetime"].values.astype("datetime64[ns]").astype(np.int64)
    for mid, grp in gym_checkins_sorted.groupby("member_id")
}

finished_streaks = streak.loc[~streak["streak_end_date"].isna()].copy()
end_ns = finished_streaks["streak_end_date"].values.astype("datetime64[ns]").astype(np.int64)
window_ns = np.int64(14) * 24 * 3600 * 1_000_000_000

recovery_flags = np.zeros(len(finished_streaks), dtype=bool)
for i, (mid, e_ns) in enumerate(zip(finished_streaks["member_id"].values, end_ns)):
    member_ns = checkin_ns_by_member.get(mid)
    if member_ns is None:
        continue
    lo = np.searchsorted(member_ns, e_ns, side="right")            # strictly after streak end
    hi = np.searchsorted(member_ns, e_ns + window_ns, side="right")
    recovery_flags[i] = (hi - lo) > 0

finished_streaks["streak_break_recovery_flag"] = recovery_flags

# data-quality note: 10 streak_id values are duplicated in streak_history_episodes_clean
# even after PR3's full-row dedup (mirrors the source_event_id issue PR6 surfaced elsewhere) —
# flagged, not silently re-fixed here; keep first occurrence so the map below stays unique.
dupe_streak_ids = finished_streaks["streak_id"].duplicated().sum()
print(f"[data quality note] streak_id duplicates in finished_streaks: {dupe_streak_ids} "
      f"(keeping first occurrence for the recovery-flag map)")

streak_recovery_map = (
    finished_streaks.drop_duplicates("streak_id", keep="first")
    .set_index("streak_id")["streak_break_recovery_flag"]
)
merged["streak_break_recovery_flag"] = merged["streak_id"].map(streak_recovery_map)
merged.loc[merged["is_streak_ongoing"] == True, "streak_break_recovery_flag"] = np.nan

print("streak_break_recovery_flag distribution (finished streaks only):")
print(finished_streaks["streak_break_recovery_flag"].value_counts(normalize=True).round(3))

[data quality note] streak_id duplicates in finished_streaks: 10 (keeping first occurrence for the recovery-flag map)
streak_break_recovery_flag distribution (finished streaks only):
streak_break_recovery_flag
False    0.897
True     0.103
Name: proportion, dtype: float64


## 7.5 Persist feature-engineered table
CSV throughout, per project convention.

In [6]:
out_path = PROCESSED_DIR / "streakforge_features.csv"
merged.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({merged.shape[0]:,} rows x {merged.shape[1]} columns)")

Saved: ..\data\processed\streakforge_features.csv  (206,732 rows x 80 columns)
